In [ ]:
# Import all required libraries for data manipulation, modeling, cross-validation, and metrics
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
import pandas as pd

# Evaluate model performance using a standard hold-out validation strategy (80/20 split)
# Load data and define features (X) and target (y)
data = pd.read_csv('/content/sample_data/mnist_train_small.csv', header=None) # MNIST dataset has no header
X = data.iloc[:, 1:] # Features are all columns except the first one
y = data.iloc[:, 0]  # Target is the first column

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

holdout_acc = accuracy_score(y_test, model.predict(X_test_s))
print(f"Hold-out accuracy: {holdout_acc:.4f}")

Hold-out accuracy: 0.8902


In [ ]:
# Evaluate model resilience using standard 5-fold cross-validation encapsulated inside a pipeline
scores_cv = cross_val_score(
    Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    X, y, cv=5, scoring='accuracy'
)
print(f"5-Fold CV: {scores_cv.mean():.4f} +/- {scores_cv.std():.4f}")

5-Fold CV: 0.8841 +/- 0.0054


In [ ]:
# Evaluate model performance using Stratified 5-fold cross-validation to maintain consistent class proportions
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_strat = cross_val_score(
    Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    X, y, cv=skf, scoring='accuracy'
)
print(f"Stratified 5-Fold CV: {scores_strat.mean():.4f} +/- {scores_strat.std():.4f}")

Stratified 5-Fold CV: 0.8867 +/- 0.0056


In [ ]:
# Demonstration of Data Leakage: Scaling the entire dataset BEFORE splitting (THIS IS WRONG ON PURPOSE)
scaler_leaked = StandardScaler()
X_scaled_all = scaler_leaked.fit_transform(X) # Fitting on ALL data includes test set distributions

X_train_leaked, X_test_leaked, y_train_l, y_test_l = train_test_split(
    X_scaled_all, y, test_size=0.2, random_state=42
)

model_leaked = LogisticRegression(max_iter=1000)
model_leaked.fit(X_train_leaked, y_train_l)

leaked_acc = accuracy_score(y_test_l, model_leaked.predict(X_test_leaked))
print(f"Leaked accuracy: {leaked_acc:.4f}")

Leaked accuracy: 0.8905


In [ ]:
# Correct manual process: Split the raw features first, then fit the scaler purely on training observations
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_correct = StandardScaler()
X_train_correct = scaler_correct.fit_transform(X_train_c)
X_test_correct = scaler_correct.transform(X_test_c) # transform only, no fit

model_correct = LogisticRegression(max_iter=1000)
model_correct.fit(X_train_correct, y_train_c)

correct_acc = accuracy_score(y_test_c, model_correct.predict(X_test_correct))
print(f"Correct accuracy: {correct_acc:.4f}")

Correct accuracy: 0.8902


In [ ]:
# Robust automation: Use a Scikit-Learn Pipeline to safely enforce correct split-then-scale operations natively
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])

pipe.fit(X_train_c, y_train_c)
pipe_acc = accuracy_score(y_test_c, pipe.predict(X_test_c))
print(f"Pipeline accuracy: {pipe_acc:.4f}")

Pipeline accuracy: 0.8902


In [ ]:
# Quantify how much the metrics were artificially inflated due to data leakage
inflation = leaked_acc - correct_acc
print(f"Accuracy inflation due to leak: {inflation:.4f}")

Accuracy inflation due to leak: 0.0002
